In [ ]:

from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
# ============================================
#  Função de normalização de strings e geração do dataset V3 - contendo apenas dados da prospecção
# ============================================


dataset = "v3"
df_treino = pd.read_csv(f"../datasets/df_treino_{dataset}.csv")
df_treino = df_treino.drop(columns= ["cavidade"])

y=df_treino["target"].copy()
X=df_treino.drop(columns=["target"]).copy()
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,

    test_size=0.2,
    random_state=42,
    stratify=y
)

df_treino.shape

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import joblib
from sasctl.pzmm import PickleModel
from sasctl.pzmm import ScoreCode
from sasctl.pzmm import JSONFiles
from sasctl.pzmm import ZipModel
from pathlib import Path

random_forest = RandomForestClassifier(
    n_estimators=1400,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

random_forest.fit(
    X_train,
    y_train
)

y_prob = random_forest.predict_proba(X_test)[:,1]


for threshold in [0.5, 0.45, 0.4, 0.35, 0.3]:

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    print(f"\nRandom forest - Dataset {dataset}\nThreshold: {threshold}")

    print(
        classification_report(
            y_test,
            y_pred
        )
    )

output_dir_model_pkl = Path("../artifacts")
output_dir_model_pkl.mkdir(
    parents=True,
    exist_ok=True
)

joblib.dump(random_forest,  output_dir_model_pkl / f"random_forest_{dataset}.pkl")

output_dir_sasmodel = Path("../sas_model")
output_dir_sasmodel.mkdir(
    parents=True,
    exist_ok=True
)

PickleModel.pickle_trained_model(
    f"Espeleo-ML-RF-{dataset}",
    random_forest,
    output_dir_sasmodel
)


sc = ScoreCode()
sc.write_score_code(
    model_prefix= f"Espeleo-ML-RF-{dataset}",
    input_data=X_train,
    target_variable="target",
    predict_method=[random_forest.predict_proba,
                    [float, float]],
    target_index=1,
    target_values=[0,1],
    score_code_path=output_dir_sasmodel,
    model_file_name=f"Espeleo-ML-RF-{dataset}.pickle",
    predict_threshold=0.4
)




model_prefix = f"Espeleo-ML-RF-{dataset}"

model_path = output_dir_sasmodel / model_prefix
model_path.mkdir(parents=True, exist_ok=True)

JSONFiles.write_var_json(
    input_data=X_train,
    is_input=True,
    json_path=model_path
)



In [ ]:
from pathlib import Path
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

from sasctl.pzmm import PickleModel
from sasctl.pzmm import ScoreCode
from sasctl.pzmm import JSONFiles
from sasctl.pzmm import ZipModel


# ============ TREINO ============
random_forest = RandomForestClassifier(
    n_estimators=1400,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

random_forest.fit(X_train, y_train)

y_prob = random_forest.predict_proba(X_test)[:, 1]

for threshold in [0.5, 0.45, 0.4, 0.35, 0.3]:

    y_pred = (y_prob >= threshold).astype(int)

    print(f"\nRandom forest - Dataset {dataset}\nThreshold: {threshold}")
    print(classification_report(y_test, y_pred))


# ============ PKL LOCAL (backup) ============
output_dir_pkl = Path("../artifacts")
output_dir_pkl.mkdir(parents=True, exist_ok=True)

joblib.dump(
    random_forest,
    output_dir_pkl / f"random_forest_{dataset}.pkl"
)


# ============ PASTA DO MODELO SAS ============
model_prefix = f"Espeleo_ML_RF_{dataset}"   # underscore, não hífen

model_path = Path("../sas_model") / model_prefix
model_path.mkdir(parents=True, exist_ok=True)


# ============ 1. PICKLE ============
PickleModel.pickle_trained_model(
    model_prefix,
    random_forest,
    model_path
)


# ============ 2. SCORE CODE ============
sc = ScoreCode()
sc.write_score_code(
    model_prefix=model_prefix,
    input_data=X_train,
    target_variable="target",
    predict_method=[
        random_forest.predict_proba,
        [float, float]
    ],
    target_index=1,
    target_values=[0, 1],
    score_code_path=model_path,
    model_file_name=f"{model_prefix}.pickle",
    predict_threshold=0.4
)


# ============ 3. inputVar.json ============
JSONFiles.write_var_json(
    input_data=X_train,
    is_input=True,
    json_path=model_path
)


# ============ 4. outputVar.json ============
output_df = pd.DataFrame({
    "I_target": [1],
    "P_0": [0.2],
    "P_1": [0.8]
})

JSONFiles.write_var_json(
    input_data=output_df,
    is_input=False,
    json_path=model_path
)


# ============ 5. ModelProperties.json ============
JSONFiles.write_model_properties_json(
    model_name=model_prefix,
    target_variable="target",
    target_values=[1, 0],
    json_path=model_path,
    model_desc="Random Forest - relevância de cavidades",
    model_algorithm="Random Forest",
    model_function="classification",
    modeler="Gabriel So"
)


# ============ 6. fileMetadata.json ============
JSONFiles.write_file_metadata_json(
    model_prefix=model_prefix,
    json_path=model_path
)


# ============ 7. ZIP FINAL ============
ZipModel.zip_files(
    model_files=model_path,
    model_prefix=model_prefix,
    is_viya4=True
)


# ============ CONFERÊNCIA ============
print("\nArquivos gerados:")
for file in sorted(model_path.iterdir()):
    print(" -", file.name)